# Music Annotation Project V8.0: Observed-Print Classifier Challenger

**Runtime:** CPU for audit/dataset construction and baseline models; GPU is optional for a later compact-CNN extension.

This is a **development-only challenger notebook**. It does not modify frozen V7.8, does not consume the frozen V7.8 independent set for training, and does not authorise production promotion.

The notebook implements the commissioning starting point:

1. build pixel/patch labels only from registered development truth;
2. restrict inference predictors to observable, truth-free inputs;
3. classify eligible observed dark pixels as **PRINT / ANNOTATION / UNCERTAIN**;
4. create page/source-group separated out-of-fold predictions;
5. apply the pre-registered safety-first decision rule;
6. compare added preservation with frozen V7.8;
7. export development evidence and a frozen-challenger candidate bundle only if all required gates pass.

**Important:** clean pages are used only to create labels and score development predictions. They must never enter classifier inference features.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn joblib matplotlib


## 1. Configuration and pre-registration constants

Edit only path overrides if your durable folder names differ. Do not change target rules, feature schema, decision thresholds or gates after inspecting development outcomes without versioning this as a new experiment.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, platform, sys, shutil
import cv2, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import confusion_matrix

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V77_ROOT=ROOT/'structural_risk_calibration_v7_7'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_0'
V75_RUN_OVERRIDE=None
V77_RUN_OVERRIDE=None
SOURCE_GROUP_MANIFEST_OVERRIDE=None  # optional CSV containing page,source_group

RANDOM_SEED=42
MASK_DILATION=2
PATCH_SIZE=33
DARK_THRESHOLD=200
TRUTH_TOLERANCE=1
MIN_COMPONENT_AREA=20
MAX_SAMPLES_PER_PAGE_PER_CLASS=25000

# Commissioning v1.1 decision posture.
PRINT_PROBABILITY_THRESHOLD=0.995
ANNOTATION_PROBABILITY_THRESHOLD=0.995
PRINT_MARGIN_OVER_ANNOTATION=0.990
OOD_TRAIN_QUANTILE=0.99
UNCERTAIN_UPPER_OPERABILITY_LIMIT=0.95

# Development gates.
MAX_FALSE_TRUE_RATIO=0.10
MAX_FALSE_TRUE_RATIO_CLUSTERED_UPPER=0.20
MAX_FALSE_PRESERVATION_RATE=0.005
MAX_FALSE_PRESERVATION_RATE_UPPER=0.010
MAX_SOURCE_FALSE_TRUE_RATIO=0.25
SPARSE_MAX_FALSE_PRESERVATION_RATE=0.0025
MIN_EXTRA_TRUE_PRINT=500
MIN_BENEFITING_PAGES=10
MIN_BENEFITING_SOURCE_GROUPS=3
CATASTROPHIC_FALSE_PIXELS=50
CATASTROPHIC_RATIO=1.0
CATASTROPHIC_MIN_FALSE_PIXELS=20

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_0_observed_print'
OUT=OUTPUT_ROOT/RUN
TABLES=OUT/'tables'; ARTEFACTS=OUT/'artefacts'; CHARTS=OUT/'charts'; AUDIT=OUT/'audit'
for p in [OUT,TABLES,ARTEFACTS,CHARTS,AUDIT]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)


## 2. Helpers, run discovery and cryptographic identity


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder):
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def object_hash(v): return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def wilson_upper(errors,n,z=1.96):
    if n==0:return np.nan
    p=errors/n; den=1+z*z/n; centre=p+z*z/(2*n); radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)); return (centre+radius)/den

def clustered_bootstrap(rows,value_fn,samples=10000,seed=42):
    pages=np.array(sorted(rows.page.unique()))
    if len(pages)==0:return np.array([])
    rng=np.random.default_rng(seed); vals=[]
    groups={p:rows[rows.page==p] for p in pages}
    for _ in range(samples):
        draw=rng.choice(pages,size=len(pages),replace=True)
        boot=pd.concat([groups[p] for p in draw],ignore_index=True)
        vals.append(value_fn(boot))
    return np.asarray(vals,float)

V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest_complete(V75_ROOT,['tables/cache_status.csv'])
V77=Path(V77_RUN_OVERRIDE) if V77_RUN_OVERRIDE else latest_complete(V77_ROOT,['tables/asymmetric_component_dataset.csv','v7_7_manifest.json'])
if V75 is None: raise RuntimeError('Complete V7.5 development run not found.')
if V77 is None: raise RuntimeError('Complete V7.7 development run not found.')
print('V7.5 development reference:',V75)
print('V7.7 development reference:',V77)


## 3. Load frozen V7.8-compatible development arrays

The notebook reuses the durable V7.5 development cache as the authoritative development source for the observed page, dilation-2 removal mask, frozen `geometry_t80` reconstruction and V7.8 route/support arrays. It fails closed if the arrays needed for challenger construction are missing or inconsistent.


In [ ]:
status=pd.read_csv(V75/'tables'/'cache_status.csv')
cache_lookup={str(r.page):Path(r.cache) for _,r in status.iterrows()}
clean_map=image_map(ROOT/'source'/'clean')
observed_map=image_map(ROOT/'aligned')

required_arrays=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
page_rows=[]; runtime={}; failures=[]
for page in sorted(set(cache_lookup)&set(clean_map)&set(observed_map)):
    try:
        with np.load(cache_lookup[page]) as z:
            missing=[k for k in required_arrays if k not in z.files]
            if missing: raise RuntimeError(f'missing arrays {missing}')
            clean=cv2.imread(str(clean_map[page]),0); observed=cv2.imread(str(observed_map[page]),0)
            if clean is None or observed is None or clean.shape!=observed.shape: raise RuntimeError('clean/observed unreadable or shape mismatch')
            arrays={k:z[k] for k in required_arrays}
        if any(arr.shape!=observed.shape for arr in arrays.values()): raise RuntimeError('cached array shape mismatch')
        mask=arrays['mask']>0
        if not np.array_equal(arrays['geometry_t80'][~mask],observed[~mask]): raise RuntimeError('V7.8 geometry changes pixels outside removal mask')
        runtime[page]={'clean':clean,'observed':observed,'mask':mask,'geometry_t80':arrays['geometry_t80'],
                       'preserve':arrays['g80_preserve']>0,'remove':arrays['g80_remove']>0,'ambiguous':arrays['g80_ambiguous']>0,
                       'horizontal':arrays['g80_horizontal']>0,'vertical':arrays['g80_vertical']>0,'compact':arrays['g80_compact']>0}
        page_rows.append({'page':page,'shape':str(observed.shape),'mask_pixels':int(mask.sum()),'cache':str(cache_lookup[page]),'cache_sha256':sha256_file(cache_lookup[page])})
    except Exception as e: failures.append({'page':page,'error':str(e)})

pd.DataFrame(page_rows).to_csv(AUDIT/'development_pages.csv',index=False)
pd.DataFrame(failures).to_csv(AUDIT/'development_load_failures.csv',index=False)
if failures: raise RuntimeError(f'Development data integrity failed on {len(failures)} page(s).')
if len(runtime)<3: raise RuntimeError('At least three valid development pages are required.')
print('Development pages:',len(runtime))


## 4. Source-group map and split key

If a development source-group manifest is available, it is used as the outer grouping key. Otherwise the notebook uses page as the grouping key and records that source-group separation could not be established from available development metadata.


In [ ]:
manifest_candidates=[]
if SOURCE_GROUP_MANIFEST_OVERRIDE: manifest_candidates.append(Path(SOURCE_GROUP_MANIFEST_OVERRIDE))
manifest_candidates += [ROOT/'source_group_manifest.csv', ROOT/'score_manifest.csv', V77/'score_manifest.csv']
source_manifest=None
for p in manifest_candidates:
    if p.exists():
        q=pd.read_csv(p)
        if {'page','source_group'}.issubset(q.columns): source_manifest=q[['page','source_group']].copy(); print('Source-group manifest:',p); break
if source_manifest is None:
    source_manifest=pd.DataFrame({'page':sorted(runtime),'source_group':sorted(runtime)})
    source_group_available=False
    print('WARNING: no development source_group manifest found; page grouping will be used.')
else:
    source_group_available=True
source_group_map=dict(zip(source_manifest.page.astype(str),source_manifest.source_group.astype(str)))
missing_groups=sorted(set(runtime)-set(source_group_map))
if missing_groups:
    raise RuntimeError(f'Source-group manifest missing development pages: {missing_groups[:20]}')


## 5. Construct truth labels

Eligible candidates are observed pixels inside the authorised mask, darker than 200, and not already preserved by frozen V7.8. Clean printed ink is derived only for development truth. A one-pixel tolerance is used around the central coordinate.

To avoid pretending that registration is perfectly exact, pixels near label disagreement are assigned **UNCERTAIN** and are not treated as definite PRINT/ANNOTATION training truth.


In [ ]:
def printed_ink(clean):
    return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def truth_maps(clean):
    ink=printed_ink(clean)
    k=np.ones((2*TRUTH_TOLERANCE+1,2*TRUTH_TOLERANCE+1),np.uint8)
    near=cv2.dilate(ink.astype(np.uint8),k)>0
    # Boundary zone becomes uncertain: local label changes within tolerance.
    er=cv2.erode(ink.astype(np.uint8),k)>0
    boundary=np.logical_xor(near,er)
    return ink,near,boundary

label_counts=[]
for page,z in runtime.items():
    ink,near,boundary=truth_maps(z['clean'])
    eligible=z['mask']&(z['observed']<DARK_THRESHOLD)&~z['preserve']
    label=np.full(z['observed'].shape,-1,np.int8) # -1 ineligible
    label[eligible&near&~boundary]=1              # PRINT
    label[eligible&~near&~boundary]=0             # ANNOTATION
    label[eligible&boundary]=2                    # UNCERTAIN truth
    z.update({'clean_ink':ink,'eligible':eligible,'truth_label':label})
    label_counts.append({'page':page,'source_group':source_group_map[page],'eligible':int(eligible.sum()),'print':int((label==1).sum()),'annotation':int((label==0).sum()),'uncertain':int((label==2).sum())})
label_df=pd.DataFrame(label_counts)
label_df.to_csv(TABLES/'label_counts_by_page.csv',index=False)
display(label_df.describe(include='all'))


## 6. Observable input allowlist

Primary V8.0 baseline features are derived only from the observed page, authorised mask and frozen V7.8 route/support context:

- 33×33 observed greyscale patch summaries;
- mask depth / mask-neighbourhood fractions;
- V7.8 horizontal, vertical and compact support fractions;
- observed edge fractions;
- background-normalised central intensity;
- frozen V7.8 preserve/remove/ambiguous neighbourhood fractions;
- fixed orientation-line responses.

No clean/truth field, page ID, source-group ID or raw x/y coordinate is passed to the estimator.


In [ ]:
def background_surface(observed,mask,sigma=45.0):
    visible=(~mask)&(observed>=200); values=observed.astype(np.float32)*visible; weights=visible.astype(np.float32)
    num=cv2.GaussianBlur(values,(0,0),sigma); den=cv2.GaussianBlur(weights,(0,0),sigma)
    surface=num/np.maximum(den,1e-4); fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)

def local_mean(a,k): return cv2.boxFilter(a.astype(np.float32),-1,(k,k),normalize=True,borderType=cv2.BORDER_REFLECT)
def extract_feature_images(z):
    obs=z['observed']; mask=z['mask']; paper=background_surface(obs,mask); norm=cv2.divide(obs,np.maximum(paper,1),scale=255)
    edge=cv2.Canny(obs,70,170)>0; depth=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5)
    dark=obs<190
    feats={
      'central_intensity':obs.astype(np.float32), 'background_normalised_intensity':norm.astype(np.float32),'mask_depth':depth.astype(np.float32),
      'patch_mean':local_mean(obs,PATCH_SIZE),'patch_dark_fraction':local_mean(dark,PATCH_SIZE),'patch_edge_fraction':local_mean(edge,PATCH_SIZE),
      'patch_mask_fraction':local_mean(mask,PATCH_SIZE),'horizontal_fraction_local':local_mean(z['horizontal'],PATCH_SIZE),
      'vertical_fraction_local':local_mean(z['vertical'],PATCH_SIZE),'compact_fraction_local':local_mean(z['compact'],PATCH_SIZE),
      'preserve_fraction_local':local_mean(z['preserve'],PATCH_SIZE),'remove_fraction_local':local_mean(z['remove'],PATCH_SIZE),
      'ambiguous_fraction_local':local_mean(z['ambiguous'],PATCH_SIZE),
    }
    # Fixed observed-only orientation responses.
    inv=(255-obs).astype(np.float32)/255.0
    feats['horizontal_line_response']=cv2.boxFilter(inv,-1,(35,1),normalize=True,borderType=cv2.BORDER_REFLECT)
    feats['vertical_line_response']=cv2.boxFilter(inv,-1,(1,15),normalize=True,borderType=cv2.BORDER_REFLECT)
    return feats

FEATURES=['central_intensity','background_normalised_intensity','mask_depth','patch_mean','patch_dark_fraction','patch_edge_fraction','patch_mask_fraction','horizontal_fraction_local','vertical_fraction_local','compact_fraction_local','preserve_fraction_local','remove_fraction_local','ambiguous_fraction_local','horizontal_line_response','vertical_line_response']
FORBIDDEN_PATTERNS=['clean','truth','outcome','failure','precision','recall','mae','source_group','page','x','y']
assert not any(any(p==c.lower() or p in c.lower().split('_') for p in FORBIDDEN_PATTERNS) for c in FEATURES)
print('Ordered feature allowlist:',FEATURES)


## 7. Sample development pixels without crossing page/source boundaries

Sampling caps are used only to keep the research notebook tractable. All sampled rows retain page/source-group identifiers **outside** the estimator matrix for grouping and reporting.


In [ ]:
rng=np.random.default_rng(RANDOM_SEED); rows=[]
for page,z in runtime.items():
    fi=extract_feature_images(z); label=z['truth_label']
    for cls,name in [(0,'ANNOTATION'),(1,'PRINT')]:
        ys,xs=np.where(label==cls); n=len(ys)
        if n==0: continue
        take=np.arange(n) if n<=MAX_SAMPLES_PER_PAGE_PER_CLASS else rng.choice(n,MAX_SAMPLES_PER_PAGE_PER_CLASS,replace=False)
        for j in take:
            y,x=int(ys[j]),int(xs[j]); row={'page':page,'source_group':source_group_map[page],'truth':cls,'truth_name':name}
            for f in FEATURES: row[f]=float(fi[f][y,x])
            rows.append(row)
samples=pd.DataFrame(rows)
if samples.empty or samples.truth.nunique()<2: raise RuntimeError('Training sample lacks both PRINT and ANNOTATION classes.')
samples.to_csv(TABLES/'development_training_sample.csv',index=False)
print(samples.groupby(['truth_name']).size())
print('Pages:',samples.page.nunique(),'source groups:',samples.source_group.nunique())


## 8. Page/source-grouped OOF model comparison and cross-fitted calibration

V8.0 begins with three conventional challengers: regularised logistic regression, random forest and histogram gradient boosting. This first notebook intentionally postpones a CNN until the audit and tabular baselines establish that the labels and gates behave sensibly.


In [ ]:
X=samples[FEATURES].replace([np.inf,-np.inf],np.nan)
y=samples.truth.astype(int)
groups=samples.source_group.astype(str) if source_group_available and samples.source_group.nunique()>=3 else samples.page.astype(str)
unique_groups=groups.nunique()
if unique_groups<3: raise RuntimeError('At least three independent page/source groups are required.')
splitter=LeaveOneGroupOut() if unique_groups<=10 else GroupKFold(n_splits=min(10,unique_groups))

models={
 'logistic':Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=3000,random_state=RANDOM_SEED))]),
 'forest':Pipeline([('impute',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=500,min_samples_leaf=5,max_features='sqrt',class_weight='balanced_subsample',random_state=RANDOM_SEED,n_jobs=-1))]),
 'hist_gradient':Pipeline([('impute',SimpleImputer(strategy='median')),('model',HistGradientBoostingClassifier(max_iter=250,max_leaf_nodes=15,min_samples_leaf=20,l2_regularization=2.0,random_state=RANDOM_SEED))]),
}

def crossfit_model(estimator):
    raw=np.full(len(samples),np.nan); fold_id=np.full(len(samples),-1,int)
    for fold,(tr,te) in enumerate(splitter.split(X,y,groups),1):
        if y.iloc[tr].nunique()<2: continue
        m=clone(estimator); m.fit(X.iloc[tr],y.iloc[tr]); raw[te]=m.predict_proba(X.iloc[te])[:,1]; fold_id[te]=fold
    # second grouped pass calibrates only from predictions outside each target group's training data
    calibrated=np.full(len(samples),np.nan)
    for tr,te in splitter.split(raw.reshape(-1,1),y,groups):
        ok=np.isfinite(raw[tr])
        if ok.sum()<20 or y.iloc[tr][ok].nunique()<2: continue
        iso=IsotonicRegression(out_of_bounds='clip').fit(raw[tr][ok],y.iloc[tr].to_numpy()[ok])
        calibrated[te]=iso.predict(raw[te])
    return raw,calibrated,fold_id

oof={}; summary=[]
for name,m in models.items():
    raw,prob,fold=crossfit_model(m); valid=np.isfinite(prob)
    if not valid.any(): continue
    pred=(prob[valid]>=0.5).astype(int); cm=confusion_matrix(y[valid],pred,labels=[0,1])
    oof[name]={'raw':raw,'prob':prob,'fold':fold}
    summary.append({'model':name,'scored':int(valid.sum()),'pages':int(samples.loc[valid,'page'].nunique()),'groups':int(groups[valid].nunique()),'tn':int(cm[0,0]),'fp':int(cm[0,1]),'fn':int(cm[1,0]),'tp':int(cm[1,1])})
summary_df=pd.DataFrame(summary); summary_df.to_csv(TABLES/'oof_model_summary.csv',index=False); display(summary_df)


## 9. Safety-first decision rule and model selection

For this binary development estimator, `p_print` is converted to three operational outputs:

- **PRINT** only when calibrated `p_print ≥ 0.995` and the PRINT-vs-ANNOTATION margin is at least 0.990;
- **ANNOTATION** only when `p_annotation ≥ 0.995`;
- otherwise **UNCERTAIN**.

This is deliberately conservative. The decision rule can later be extended with ensemble disagreement and explicit OOD rejection once the first label/model audit is complete.


In [ ]:
def decide_from_probability(p):
    p=np.asarray(p,float); pa=1-p; out=np.full(len(p),'UNCERTAIN',object)
    out[pa>=ANNOTATION_PROBABILITY_THRESHOLD]='ANNOTATION'
    print_ok=(p>=PRINT_PROBABILITY_THRESHOLD)&((p-pa)>=PRINT_MARGIN_OVER_ANNOTATION)
    out[print_ok]='PRINT'
    return out

model_eval=[]
for name,v in oof.items():
    prob=v['prob']; valid=np.isfinite(prob); decisions=np.full(len(samples),'UNSCORED',object); decisions[valid]=decide_from_probability(prob[valid])
    false_print=int(np.sum(valid & (decisions=='PRINT') & (samples.truth.to_numpy()==0)))
    true_print=int(np.sum(valid & (decisions=='PRINT') & (samples.truth.to_numpy()==1)))
    ratio=false_print/max(true_print,1); utility=true_print-5*false_print
    uncertain=float(np.mean(decisions[valid]=='UNCERTAIN'))
    model_eval.append({'model':name,'true_print_sample':true_print,'false_annotation_sample':false_print,'false_true_ratio_sample':ratio,'utility_sample':utility,'uncertain_rate_sample':uncertain})
    samples[f'p_print_{name}']=prob; samples[f'decision_{name}']=decisions
model_eval=pd.DataFrame(model_eval).sort_values(['false_true_ratio_sample','utility_sample'],ascending=[True,False])
model_eval.to_csv(TABLES/'oof_decision_model_comparison.csv',index=False); display(model_eval)
SELECTED_MODEL=str(model_eval.iloc[0].model)
print('Provisional selected model for full-pixel OOF scoring:',SELECTED_MODEL)


## 10. Full-pixel page-grouped OOF scoring

The selected model is refit separately inside each outer fold and applied to **all eligible pixels on the held-out pages/source groups**. This stage produces the actual development safety evidence against frozen V7.8 rather than relying on the capped training sample.


In [ ]:
# Build a calibration function for each outer fold from training-group sample OOF scores only.
full_records=[]
base_model=models[SELECTED_MODEL]
for fold,(tr,te) in enumerate(splitter.split(X,y,groups),1):
    train_groups=set(groups.iloc[tr]); test_groups=set(groups.iloc[te])
    train_pages=sorted(set(samples.loc[tr,'page'])); test_pages=sorted(set(samples.loc[te,'page']))
    # Fit estimator on training sample.
    m=clone(base_model); m.fit(X.iloc[tr],y.iloc[tr])
    # Calibration from selected model's existing OOF probabilities restricted to training rows.
    raw_train=oof[SELECTED_MODEL]['raw'][tr]; ok=np.isfinite(raw_train)
    if ok.sum()<20 or y.iloc[tr][ok].nunique()<2: raise RuntimeError(f'Fold {fold}: insufficient calibration rows')
    iso=IsotonicRegression(out_of_bounds='clip').fit(raw_train[ok],y.iloc[tr].to_numpy()[ok])
    for page in test_pages:
        z=runtime[page]; fi=extract_feature_images(z); ys,xs=np.where(z['eligible'])
        if len(ys)==0: continue
        F=pd.DataFrame({f:fi[f][ys,xs].astype(float) for f in FEATURES})
        raw=m.predict_proba(F)[:,1]; prob=iso.predict(raw); decision=decide_from_probability(prob)
        truth=z['truth_label'][ys,xs]
        baseline=z['geometry_t80'][ys,xs]; observed=z['observed'][ys,xs]
        added=(decision=='PRINT') & (baseline!=observed)
        for i in range(len(ys)):
            if truth[i] not in (0,1,2): continue
            full_records.append({'fold':fold,'page':page,'source_group':source_group_map[page],'y':int(ys[i]),'x':int(xs[i]),'truth':int(truth[i]),'p_print':float(prob[i]),'decision':decision[i],
                                 'baseline_pixel':int(baseline[i]),'observed_pixel':int(observed[i]),'added_preservation':bool(added[i])})
full=pd.DataFrame(full_records)
full.to_parquet(TABLES/'full_pixel_oof_predictions.parquet',index=False)
print('Full OOF eligible rows:',len(full))


## 11. Development metrics and gates


In [ ]:
q=full[full.truth.isin([0,1])].copy()
q['extra_true']=q.added_preservation&(q.truth==1)
q['extra_false']=q.added_preservation&(q.truth==0)
true_count=int(q.extra_true.sum()); false_count=int(q.extra_false.sum()); added=int(q.added_preservation.sum())
false_true=false_count/max(true_count,1); false_pres=false_count/max(added,1); utility=true_count-5*false_count

# Catastrophic runs are approximated at page connected-run level from added false/true pixels.
catastrophic=[]
for page,g in q[q.added_preservation].groupby('page'):
    z=runtime[page]; mask=np.zeros(z['observed'].shape,np.uint8)
    mask[g.y.to_numpy(),g.x.to_numpy()]=1
    n,lab,stats,_=cv2.connectedComponentsWithStats(mask,8)
    lookup={(int(r.y),int(r.x)):r for _,r in g.iterrows()}
    for cid in range(1,n):
        ys,xs=np.where(lab==cid); vals=[lookup[(int(y0),int(x0))] for y0,x0 in zip(ys,xs)]
        f=sum(int(v.extra_false) for v in vals); t=sum(int(v.extra_true) for v in vals); ratio=f/max(t,1)
        if f>=CATASTROPHIC_FALSE_PIXELS or (f>=CATASTROPHIC_MIN_FALSE_PIXELS and ratio>CATASTROPHIC_RATIO): catastrophic.append({'page':page,'component':cid,'false':f,'true':t,'ratio':ratio})
catastrophic_df=pd.DataFrame(catastrophic); catastrophic_df.to_csv(TABLES/'catastrophic_components.csv',index=False)

ratio_boot=clustered_bootstrap(q,lambda d:int(d.extra_false.sum())/max(int(d.extra_true.sum()),1))
utility_boot=clustered_bootstrap(q,lambda d:int(d.extra_true.sum())-5*int(d.extra_false.sum()))
false_pres_upper=wilson_upper(false_count,added)
ratio_upper=float(np.nanpercentile(ratio_boot,97.5)) if len(ratio_boot) else np.nan
utility_lower=float(np.nanpercentile(utility_boot,2.5)) if len(utility_boot) else np.nan
uncertain_rate=float((q.decision=='UNCERTAIN').mean())
benefit_pages=int(q.loc[q.extra_true].page.nunique()); benefit_groups=int(q.loc[q.extra_true].source_group.nunique())

by_group=q.groupby('source_group').agg(extra_true=('extra_true','sum'),extra_false=('extra_false','sum'),added=('added_preservation','sum')).reset_index()
by_group['false_true_ratio']=by_group.extra_false/by_group.extra_true.clip(lower=1)
by_group['false_preservation_rate']=by_group.extra_false/by_group.added.clip(lower=1)
by_group.to_csv(TABLES/'development_results_by_source_group.csv',index=False)

gates={
 'D1_integrity':len(failures)==0,
 'D2_immutability':True, # challenger writes no page output and scores only within eligible mask rows
 'D3_catastrophic_safety':len(catastrophic_df)==0,
 'D4_false_preservation_ratio':false_true<=MAX_FALSE_TRUE_RATIO and ratio_upper<=MAX_FALSE_TRUE_RATIO_CLUSTERED_UPPER,
 'D5_pixel_safety':false_pres<=MAX_FALSE_PRESERVATION_RATE and false_pres_upper<=MAX_FALSE_PRESERVATION_RATE_UPPER,
 'D6_positive_utility':utility>0 and utility_lower>0,
 'D7_nontrivial_benefit':true_count>=MIN_EXTRA_TRUE_PRINT and benefit_pages>=MIN_BENEFITING_PAGES and benefit_groups>=MIN_BENEFITING_SOURCE_GROUPS,
 'D8_source_stability':bool(len(by_group)>0 and (by_group.false_true_ratio<=MAX_SOURCE_FALSE_TRUE_RATIO).all()),
 'D9_sparse_hard_negative':'NOT_EVALUATED_IN_V8_0_STARTER',
 'D10_uncertainty_operability':uncertain_rate<=UNCERTAIN_UPPER_OPERABILITY_LIMIT,
 'D11_baseline_dominance':'REQUIRES_PRE_REGISTERED_NONLEARNED_BASELINE_TABLE',
 'D12_visual_audit':'REQUIRES_HUMAN_REVIEW',
}
metrics={'extra_true_print':true_count,'extra_false_annotation':false_count,'false_true_ratio':false_true,'false_true_ratio_clustered_95_upper':ratio_upper,'false_preservation_rate':false_pres,'false_preservation_wilson95_upper':false_pres_upper,'net_safety_utility':utility,'utility_clustered_95_lower':utility_lower,'uncertain_rate':uncertain_rate,'benefiting_pages':benefit_pages,'benefiting_source_groups':benefit_groups,'catastrophic_components':len(catastrophic_df)}
(TABLES/'development_metrics.json').write_text(json.dumps(metrics,indent=2))
(TABLES/'development_gates.json').write_text(json.dumps(gates,indent=2))
print(json.dumps(metrics,indent=2)); print(json.dumps(gates,indent=2))


## 12. Visual-audit queues

This exports candidate rows for the most beneficial, most false-heavy and most uncertain cases. Image contact-sheet generation can be added after confirming the development label audit because this starter deliberately avoids creating a polished visual layer before label integrity is established.


In [ ]:
false_heavy=q[q.extra_false].sort_values('p_print',ascending=False).head(200)
beneficial=q[q.extra_true].sort_values('p_print',ascending=False).head(200)
uncertain=q[q.decision=='UNCERTAIN'].assign(distance=lambda d:(d.p_print-0.5).abs()).sort_values('distance').head(200)
false_heavy.to_csv(TABLES/'visual_audit_false_heavy.csv',index=False)
beneficial.to_csv(TABLES/'visual_audit_beneficial.csv',index=False)
uncertain.to_csv(TABLES/'visual_audit_uncertain.csv',index=False)
print('Visual audit queues exported.')


## 13. Freeze candidate artefacts only after the development review is complete

This cell intentionally refuses to create a frozen challenger unless automated gates D1-D8 and D10 pass and the operator explicitly records completion of D9, D11 and D12. That prevents this starter notebook from silently treating incomplete evidence as promotion-ready.


In [ ]:
D9_CONFIRMED=False
D11_CONFIRMED=False
D12_CONFIRMED=False
CREATE_FROZEN_CHALLENGER=False

required_auto=[k for k,v in gates.items() if isinstance(v,bool) and k not in []]
auto_pass=all(gates[k] for k in required_auto)
manual_pass=D9_CONFIRMED and D11_CONFIRMED and D12_CONFIRMED
if CREATE_FROZEN_CHALLENGER:
    if not auto_pass: raise RuntimeError(f'Automated development gates not all passed: {gates}')
    if not manual_pass: raise RuntimeError('D9, D11 and D12 must be explicitly completed before freezing.')
    final_model=clone(models[SELECTED_MODEL]); final_model.fit(X,y)
    # Final calibrator is fit on all valid OOF raw scores only.
    raw=oof[SELECTED_MODEL]['raw']; ok=np.isfinite(raw)
    calibrator=IsotonicRegression(out_of_bounds='clip').fit(raw[ok],y.to_numpy()[ok])
    joblib.dump(final_model,ARTEFACTS/'observed_print_model.joblib')
    joblib.dump(calibrator,ARTEFACTS/'isotonic_calibrator.joblib')
    schema={'ordered_features':FEATURES,'patch_size':PATCH_SIZE,'dark_threshold':DARK_THRESHOLD,'truth_tolerance':TRUTH_TOLERANCE}
    (ARTEFACTS/'predictor_schema.json').write_text(json.dumps(schema,indent=2))
    policy={'version':'8.0','classes':['PRINT','ANNOTATION','UNCERTAIN'],'p_print_threshold':PRINT_PROBABILITY_THRESHOLD,'p_annotation_threshold':ANNOTATION_PROBABILITY_THRESHOLD,'print_margin':PRINT_MARGIN_OVER_ANNOTATION,'development_metrics':metrics,'development_gates':gates}
    policy['model_sha256']=sha256_file(ARTEFACTS/'observed_print_model.joblib'); policy['calibrator_sha256']=sha256_file(ARTEFACTS/'isotonic_calibrator.joblib'); policy['schema_sha256']=sha256_file(ARTEFACTS/'predictor_schema.json'); policy['policy_id']=object_hash(policy)
    (ARTEFACTS/'policy.json').write_text(json.dumps(policy,indent=2))
    print('FROZEN CHALLENGER:',policy['policy_id'])
else:
    print('No challenger frozen. This is the correct default for the first V8.0 audit/development run.')


## 14. Run manifest and next-step checklist

Before the first meaningful development run, confirm:

- the exact V7.5 development cache/run to use;
- whether a development `page → source_group` manifest already exists;
- that the clean/observed registration and cached V7.8-compatible arrays are the intended authoritative development artefacts;
- the chosen hard-negative/sparse-print development roster for D9;
- the pre-registered non-learned comparator for D11;
- the human visual-review completion record for D12;
- a new untouched independent cohort is reserved before any production claim.

The frozen V7.8 independent set must remain outside training if it is retained as a permanent regression set.


In [ ]:
manifest={
 'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.0-starter','development_only':True,
 'v7_5_reference':str(V75),'v7_7_reference':str(V77),'pages':len(runtime),'source_group_manifest_available':source_group_available,
 'feature_allowlist':FEATURES,'selected_model':SELECTED_MODEL if 'SELECTED_MODEL' in globals() else None,
 'decision_thresholds':{'print':PRINT_PROBABILITY_THRESHOLD,'annotation':ANNOTATION_PROBABILITY_THRESHOLD,'print_margin':PRINT_MARGIN_OVER_ANNOTATION},
 'gates':gates if 'gates' in globals() else None,'software':{'python':sys.version,'platform':platform.platform(),'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__},
 'frozen_v7_8_independent_used_for_training':False,'automatic_production_promotion':False,
}
(OUT/'v8_0_run_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
print('Manifest:',OUT/'v8_0_run_manifest.json')
print('V8.0 starter complete. Review audit tables before any freeze.')
